# Hyperparameter Tuning Notebook (MNIST + CIFAR-10)

This notebook runs a **10-combination hyperparameter search** across three MLP architectures:
- Shallow: `[128]`
- Medium: `[512, 256, 128]`
- Deep: `[1024, 512, 256, 128, 64]`

It uses your project modules:
- `dataset_loaders.py`
- `model.py`
- `train.py`

Preprocessing in `dataset_loaders.py` now applies `ToTensor()` plus dataset-specific standard normalization for both MNIST and CIFAR-10.

In [1]:
!git clone https://github.com/bnguyen1212/CS4375-Project3-BHN220001.git
%cd CS4375-Project3-BHN220001

Cloning into 'CS4375-Project3-BHN220001'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 9 (delta 0), reused 9 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 9.28 KiB | 3.09 MiB/s, done.
/content/CS4375-Project3-BHN220001


In [2]:
# If needed, run this once in notebook runtime:
# %pip install -r requirements.txt

import os
import sys
import time
import random
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import ConcatDataset, DataLoader

try:
    from dataset_loaders import load_mnist, load_cifar10
    from model import MLP
    from train import train_model, evaluate
except ModuleNotFoundError as e:
    print(f'Import error: {e}')
    print(f'Current working directory: {os.getcwd()}')
    print('Top sys.path entries:')
    for p in sys.path[:5]:
        print(' -', p)
    print('Hint: run Cell 2 first and ensure it prints the correct project root.')
    raise

In [3]:
# Reproducibility + device
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [4]:
# Experiment settings
EPOCHS = 20
PATIENCE = 5
NUM_WORKERS = 0

ARCHITECTURES = {
    "shallow": [128],
    "medium": [512, 256, 128],
    "deep": [1024, 512, 256, 128, 64],
}

# 10 meaningful combinations from:
# LR {0.01, 0.001, 0.0001}
# Batch size {32, 64, 128}
# Optimizer {sgd, adam}
# Dropout {0.2, 0.5}
SEARCH_CONFIGS = [
    {"lr": 1e-2,  "batch_size": 64,  "optimizer": "sgd",  "dropout": 0.2},
    {"lr": 1e-3,  "batch_size": 32,  "optimizer": "adam", "dropout": 0.2},
    {"lr": 1e-4,  "batch_size": 128, "optimizer": "adam", "dropout": 0.5},
    {"lr": 1e-2,  "batch_size": 128, "optimizer": "sgd",  "dropout": 0.5},
    {"lr": 1e-3,  "batch_size": 64,  "optimizer": "sgd",  "dropout": 0.2},
    {"lr": 1e-4,  "batch_size": 32,  "optimizer": "sgd",  "dropout": 0.2},
    {"lr": 1e-3,  "batch_size": 128, "optimizer": "adam", "dropout": 0.5},
    {"lr": 1e-2,  "batch_size": 32,  "optimizer": "adam", "dropout": 0.2},
    {"lr": 1e-4,  "batch_size": 64,  "optimizer": "adam", "dropout": 0.2},
    {"lr": 1e-3,  "batch_size": 64,  "optimizer": "sgd",  "dropout": 0.5},
]

print(f"Architectures: {list(ARCHITECTURES.keys())}")
print(f"#search configs: {len(SEARCH_CONFIGS)}")

Architectures: ['shallow', 'medium', 'deep']
#search configs: 10


In [5]:
# Helpers
def get_loaders(dataset_name: str, batch_size: int):
    if dataset_name == "mnist":
        return load_mnist(batch_size=batch_size, num_workers=NUM_WORKERS, split_seed=SEED)
    if dataset_name == "cifar10":
        return load_cifar10(batch_size=batch_size, num_workers=NUM_WORKERS, split_seed=SEED)
    raise ValueError("dataset_name must be 'mnist' or 'cifar10'")

def build_mlp(dataset_name: str, hidden_dims, dropout: float):
    if dataset_name == "mnist":
        input_dim = 28 * 28
    elif dataset_name == "cifar10":
        input_dim = 32 * 32 * 3
    else:
        raise ValueError("dataset_name must be 'mnist' or 'cifar10'")

    return MLP(input_dim=input_dim, hidden_dims=hidden_dims, dropout=dropout, num_classes=10)

In [6]:
# Main tuning loop split by dataset so each run can be executed in separate cells
all_results = []
best_models = {}
best_histories = {}
run_idx = 0
total_runs = 2 * len(ARCHITECTURES) * len(SEARCH_CONFIGS)

def run_tuning_for_dataset(dataset_name):
    global run_idx

    for arch_name, hidden_dims in ARCHITECTURES.items():
        best_key = (dataset_name, arch_name)
        best_val_acc = -1.0
        best_model = None
        best_history = None

        for cfg_i, cfg in enumerate(SEARCH_CONFIGS, start=1):
            run_idx += 1
            print("=" * 90)
            print(
                f"Run {run_idx}/{total_runs} | dataset={dataset_name} | arch={arch_name} "
                f"| cfg={cfg_i}/10 | {cfg}"
            )

            train_loader, val_loader, test_loader = get_loaders(dataset_name, cfg["batch_size"])
            model = build_mlp(dataset_name, hidden_dims, dropout=cfg["dropout"])

            train_cfg = {
                "lr": cfg["lr"],
                "optimizer": cfg["optimizer"],
                "epochs": EPOCHS,
                "patience": PATIENCE,
            }

            trained_model, history, runtime_sec = train_model(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                config=train_cfg,
                device=device,
            )

            final_val_acc = history["val_acc"][-1] if history["val_acc"] else 0.0
            best_epoch_val_acc = max(history["val_acc"]) if history["val_acc"] else 0.0

            all_results.append({
                "dataset": dataset_name,
                "architecture": arch_name,
                "hidden_dims": str(hidden_dims),
                "config_id": cfg_i,
                "lr": cfg["lr"],
                "batch_size": cfg["batch_size"],
                "optimizer": cfg["optimizer"],
                "dropout": cfg["dropout"],
                "epochs_ran": len(history["train_loss"]),
                "final_train_loss": history["train_loss"][-1] if history["train_loss"] else None,
                "final_val_loss": history["val_loss"][-1] if history["val_loss"] else None,
                "final_val_acc": final_val_acc,
                "best_val_acc": best_epoch_val_acc,
                "runtime_sec": runtime_sec,
            })

            if best_epoch_val_acc > best_val_acc:
                best_val_acc = best_epoch_val_acc
                best_model = deepcopy(trained_model)
                best_history = deepcopy(history)

        best_models[best_key] = best_model
        best_histories[best_key] = best_history

    print(f"\nCompleted tuning for {dataset_name}.")

print("Run the next two cells separately: one for MNIST and one for CIFAR-10.")

Run the next two cells separately: one for MNIST and one for CIFAR-10.


In [ ]:
# Run tuning for MNIST only
run_tuning_for_dataset("mnist")

In [ ]:
# Run tuning for CIFAR-10 only
run_tuning_for_dataset("cifar10")

print("\nTuning loop complete.")

In [ ]:
# Summarize tuning runs and keep only the best config per dataset/architecture
results_df = pd.DataFrame(all_results)

best_cfg_df = (
    results_df
    .sort_values("best_val_acc", ascending=False)
    .groupby(["dataset", "architecture"], as_index=False)
    .first()
    .sort_values(["dataset", "architecture"])
    .reset_index(drop=True)
)

print(f"Total completed runs: {len(results_df)}")
print("Best validation config per dataset+architecture:")
display(
    best_cfg_df[[
        "dataset", "architecture", "lr", "batch_size",
        "optimizer", "dropout", "best_val_acc", "epochs_ran"
    ]]
)

In [ ]:
# Retrain selected configs on full train+validation, then evaluate on test
criterion = nn.CrossEntropyLoss()
retrained_models = {}
test_rows = []

# Best configs to use for final retraining/evaluation
BEST_RETRAIN_CONFIGS = {
    "cifar10": {"lr": 0.0001, "batch_size": 64, "optimizer": "adam", "dropout": 0.2},
    "mnist": {"lr": 0.001, "batch_size": 32, "optimizer": "adam", "dropout": 0.2},
}

for dataset_name in ["mnist", "cifar10"]:
    cfg = BEST_RETRAIN_CONFIGS[dataset_name]
    for arch_name, hidden_dims in ARCHITECTURES.items():

        train_loader, val_loader, test_loader = get_loaders(dataset_name, cfg["batch_size"])
        full_train_dataset = ConcatDataset([train_loader.dataset, val_loader.dataset])
        full_train_loader = DataLoader(
            full_train_dataset,
            batch_size=cfg["batch_size"],
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=torch.cuda.is_available(),
        )

        model = build_mlp(dataset_name, hidden_dims, dropout=cfg["dropout"])
        retrain_cfg = {
            "lr": cfg["lr"],
            "optimizer": cfg["optimizer"],
            "epochs": EPOCHS,
        }

        retrained_model, retrain_history, retrain_runtime_sec = train_model(
            model=model,
            train_loader=full_train_loader,
            val_loader=full_train_loader,
            config=retrain_cfg,
            device=device,
        )

        test_loss, test_acc = evaluate(retrained_model, test_loader, criterion, device)
        retrained_models[(dataset_name, arch_name)] = retrained_model

        test_rows.append({
            "dataset": dataset_name,
            "architecture": arch_name,
            "lr": cfg["lr"],
            "batch_size": cfg["batch_size"],
            "optimizer": cfg["optimizer"],
            "dropout": cfg["dropout"],
            "best_val_acc": float("nan"),
            "retrain_epochs": len(retrain_history["train_loss"]),
            "retrain_runtime_sec": retrain_runtime_sec,
            "test_loss": test_loss,
            "test_acc": test_acc,
        })

test_df = pd.DataFrame(test_rows).sort_values(["dataset", "architecture"]).reset_index(drop=True)
print("Final test performance after full train+validation retraining:")
display(test_df)

In [ ]:
# Save artifacts
os.makedirs("results", exist_ok=True)
results_path = os.path.join("results", "all_tuning_results.csv")
best_cfg_path = os.path.join("results", "best_validation_configs.csv")
test_path = os.path.join("results", "best_models_test_results.csv")

results_df.to_csv(results_path, index=False)
best_cfg_df.to_csv(best_cfg_path, index=False)
test_df.to_csv(test_path, index=False)

print("Saved:")
print(f"- {results_path}")
print(f"- {best_cfg_path}")
print(f"- {test_path}")